# Descarte Fácil — Machine Learning

**Pergunta:** é possível prever a quantidade de solicitações de remoção de grandes objetos por subprefeitura e mês usando o histórico público do SP156?

Fonte: [Dados do SP156 — Prefeitura de São Paulo](https://dados.prefeitura.sp.gov.br/dataset/dados-do-sp156).

> O SP156 representa solicitações registradas, não todos os descartes irregulares existentes. Confira o dicionário da extração e não interprete o resultado como previsão de coleta por rua.

## 1. Instalação e carregamento

Baixe o CSV na página oficial do SP156 e informe o caminho em `CSV_PATH`. No Google Colab, se o arquivo não estiver no caminho, o notebook abrirá a opção de upload.

In [ ]:
%pip -q install pandas numpy matplotlib scikit-learn xgboost

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import ParameterGrid
from xgboost import XGBRegressor

CSV_PATH = Path("../data/sp156.csv")  # ajuste para o local do arquivo baixado
if not CSV_PATH.exists():
    try:
        from google.colab import files
        uploaded = files.upload()
        CSV_PATH = Path(next(iter(uploaded)))
    except ImportError:
        raise FileNotFoundError("Coloque o CSV em ml/data/sp156.csv ou ajuste CSV_PATH.")

df = pd.read_csv(CSV_PATH, sep=None, engine="python", low_memory=False, encoding_errors="replace")
print("Dimensões:", df.shape)
print("Colunas:", df.columns.tolist())
display(df.head())

## 2. Mapeamento das colunas

Confira os nomes abaixo e ajuste as três variáveis para corresponder exatamente à extração e ao dicionário oficial.

In [ ]:
# AJUSTE conforme df.columns e o dicionário da base
DATE_COL = "data_abertura"
SERVICE_COL = "servico"
REGION_COL = "subprefeitura"

for col in [DATE_COL, SERVICE_COL, REGION_COL]:
    if col not in df.columns:
        raise KeyError(f"Coluna {col!r} não encontrada. Colunas disponíveis: {df.columns.tolist()}")

print(df[SERVICE_COL].astype(str).value_counts().head(40).to_string())

## 3. Seleção do serviço e agregação mensal

Confirme o rótulo do serviço na tabela anterior. Se necessário, altere `SERVICE_TERMS` para a nomenclatura exata da sua extração.

In [ ]:
SERVICE_TERMS = ["remoção de grandes objetos", "remocao de grandes objetos"]

work = df.copy()
work[DATE_COL] = pd.to_datetime(work[DATE_COL], errors="coerce", dayfirst=True)
work = work.dropna(subset=[DATE_COL, SERVICE_COL, REGION_COL])

service = work[SERVICE_COL].astype(str).str.lower()
mask = service.apply(lambda value: any(term in value for term in SERVICE_TERMS))
selected = work.loc[mask].copy()

if selected.empty:
    raise ValueError("Nenhum registro selecionado. Ajuste SERVICE_TERMS conforme os rótulos impressos.")

selected["mes"] = selected[DATE_COL].dt.to_period("M").dt.to_timestamp()
monthly = (selected.groupby([REGION_COL, "mes"]).size()
           .rename("y").reset_index().rename(columns={REGION_COL: "regiao"}))

# Painel balanceado: meses sem solicitação observada recebem zero.
regions = monthly["regiao"].unique()
months = pd.date_range(monthly["mes"].min(), monthly["mes"].max(), freq="MS")
panel = pd.MultiIndex.from_product([regions, months], names=["regiao", "mes"]).to_frame(index=False)
panel = panel.merge(monthly, on=["regiao", "mes"], how="left")
panel["y"] = panel["y"].fillna(0).astype(int)

print("Período:", panel.mes.min(), "até", panel.mes.max(), "| observações:", len(panel))
display(panel.head())

## 4. Engenharia de atributos e separação temporal

As variáveis históricas usam somente meses anteriores. Os 20% finais dos meses serão reservados para teste. Dentro do período de treino, os meses mais recentes serão usados para selecionar hiperparâmetros, evitando ajustar os modelos diretamente no teste final.

In [ ]:
panel = panel.sort_values(["regiao", "mes"]).copy()
panel["ano"] = panel["mes"].dt.year
panel["mes_num"] = panel["mes"].dt.month
panel["trimestre"] = panel["mes"].dt.quarter

for lag in [1, 2, 3, 6, 12]:
    panel[f"lag_{lag}"] = panel.groupby("regiao")["y"].shift(lag)

panel["media_movel_3"] = panel.groupby("regiao")["y"].transform(
    lambda s: s.shift(1).rolling(3).mean()
)

model_df = panel.dropna().copy()
X = pd.get_dummies(
    model_df[["regiao", "ano", "mes_num", "trimestre", "lag_1", "lag_2",
              "lag_3", "lag_6", "lag_12", "media_movel_3"]],
    columns=["regiao"], dtype=int
)
y = model_df["y"]

unique_months = np.sort(model_df["mes"].unique())
test_cut = unique_months[max(1, int(len(unique_months) * 0.8))]
train_mask = model_df["mes"] < test_cut
test_mask = model_df["mes"] >= test_cut

train_months = np.sort(model_df.loc[train_mask, "mes"].unique())
val_cut = train_months[max(1, int(len(train_months) * 0.8))]
fit_mask = model_df["mes"] < val_cut
val_mask = (model_df["mes"] >= val_cut) & train_mask

print("Início do teste:", test_cut)
print("Treino para ajuste:", fit_mask.sum(), "| validação:", val_mask.sum(), "| teste final:", test_mask.sum())

## 5. Seleção de hiperparâmetros

Os modelos são comparados na validação temporal interna ao treino. O teste final permanece separado para a avaliação de desempenho após a seleção.

In [ ]:
def mae(y_true, pred):
    return mean_absolute_error(y_true, pred)

X_fit, y_fit = X.loc[fit_mask], y.loc[fit_mask]
X_val, y_val = X.loc[val_mask], y.loc[val_mask]

if len(X_fit) == 0 or len(X_val) == 0 or len(X.loc[test_mask]) == 0:
    raise ValueError("Histórico insuficiente para divisão temporal. Confira o período disponível.")

rf_grid = list(ParameterGrid({
    "n_estimators": [200, 400],
    "max_depth": [8, None],
    "min_samples_leaf": [1, 3]
}))
best_rf, best_rf_params, best_rf_mae = None, None, float("inf")

for params in rf_grid:
    model = RandomForestRegressor(random_state=42, n_jobs=-1, **params)
    model.fit(X_fit, y_fit)
    score = mae(y_val, model.predict(X_val))
    if score < best_rf_mae:
        best_rf, best_rf_params, best_rf_mae = model, params, score

xgb_grid = list(ParameterGrid({
    "n_estimators": [200, 400],
    "max_depth": [3, 5],
    "learning_rate": [0.03, 0.08]
}))
best_xgb, best_xgb_params, best_xgb_mae = None, None, float("inf")

for params in xgb_grid:
    model = XGBRegressor(
        random_state=42, objective="reg:squarederror", n_jobs=-1, **params
    )
    model.fit(X_fit, y_fit)
    score = mae(y_val, model.predict(X_val))
    if score < best_xgb_mae:
        best_xgb, best_xgb_params, best_xgb_mae = model, params, score

print("Melhor Random Forest:", best_rf_params, "| MAE validação:", round(best_rf_mae, 3))
print("Melhor XGBoost:", best_xgb_params, "| MAE validação:", round(best_xgb_mae, 3))

## 6. Treinamento final e avaliação no teste

Após selecionar os hiperparâmetros, os modelos são treinados novamente com todo o período de treino (incluindo a validação) e avaliados uma única vez no período de teste.

In [ ]:
X_train, y_train = X.loc[train_mask], y.loc[train_mask]
X_test, y_test = X.loc[test_mask], y.loc[test_mask]

rf_final = RandomForestRegressor(random_state=42, n_jobs=-1, **best_rf_params)
rf_final.fit(X_train, y_train)

xgb_final = XGBRegressor(
    random_state=42, objective="reg:squarederror", n_jobs=-1, **best_xgb_params
)
xgb_final.fit(X_train, y_train)

baseline_pred = model_df.loc[test_mask, "lag_1"].to_numpy()
rf_pred = rf_final.predict(X_test)
xgb_pred = xgb_final.predict(X_test)

def metric_row(name, truth, pred, params="—"):
    return {
        "modelo": name,
        "MAE": mean_absolute_error(truth, pred),
        "RMSE": mean_squared_error(truth, pred) ** 0.5,
        "R2": r2_score(truth, pred),
        "hiperparametros": str(params)
    }

results = pd.DataFrame([
    metric_row("Baseline (mês anterior)", y_test, baseline_pred),
    metric_row("Random Forest", y_test, rf_pred, best_rf_params),
    metric_row("XGBoost", y_test, xgb_pred, best_xgb_params)
]).sort_values("MAE")

display(results.round(3))

In [ ]:
predictions = {
    "Baseline (mês anterior)": baseline_pred,
    "Random Forest": rf_pred,
    "XGBoost": xgb_pred
}
best_name = results.iloc[0]["modelo"]
best_pred = predictions[best_name]

plt.figure(figsize=(6, 5))
plt.scatter(y_test, best_pred, alpha=0.35)
upper = max(float(y_test.max()), float(np.max(best_pred)))
plt.plot([0, upper], [0, upper], "--")
plt.xlabel("Solicitações reais")
plt.ylabel("Solicitações previstas")
plt.title(f"Real x previsto — {best_name}")
plt.tight_layout()
plt.show()

if best_name == "Random Forest":
    importance = pd.Series(rf_final.feature_importances_, index=X.columns)
elif best_name == "XGBoost":
    importance = pd.Series(xgb_final.feature_importances_, index=X.columns)
else:
    importance = None

if importance is not None:
    importance.nlargest(15).sort_values().plot(
        kind="barh", figsize=(8, 5), title="15 atributos mais importantes"
    )
    plt.tight_layout()
    plt.show()

print("Melhor modelo por MAE no teste:", best_name)

## 7. Conclusão e interpretação

Preencha o relatório com os valores da tabela de métricas. Discuta:

- Se Random Forest ou XGBoost superou a linha de base.
- Qual modelo teve menor MAE no teste.
- Quais atributos foram mais importantes.
- Se os erros são aceitáveis para o uso proposto.

**Interpretação cidadã:** resultados positivos podem ajudar a reconhecer regiões e períodos com maior demanda registrada e orientar comunicação e planejamento. A previsão não substitui o calendário oficial do Cata-Bagulho e não garante coleta em um endereço.

**Limitações:** subnotificação, possíveis solicitações duplicadas, mudanças na classificação dos serviços e granularidade por subprefeitura. Não extrapole os resultados para CEPs ou ruas individuais.